In [ ]:
import numpy as np

from pathlib import Path

from PIL import Image

import torch
import torchvision.transforms as transforms

from torchvision import models

from tqdm import tqdm

In [ ]:
resnet = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

feature_extractor = torch.nn.Sequential(
    *list(resnet.children())[:-1]
)

feature_extractor.eval()

print("Feature Extractor Ready")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
def get_embedding(image_path):

    img = Image.open(
        image_path
    ).convert("RGB")

    img_tensor = transform(img)

    img_tensor = img_tensor.unsqueeze(0)

    with torch.no_grad():

        embedding = feature_extractor(
            img_tensor
        )

    embedding = embedding.squeeze()

    return embedding.numpy()

In [ ]:
DATASET_PATH = Path("../dataset")

image_paths = []

labels = []

for crop in DATASET_PATH.iterdir():

    train_folder = crop / "Train"

    if not train_folder.exists():
        continue

    for disease in train_folder.iterdir():

        if not disease.is_dir():
            continue

        images = [
            img
            for img in disease.glob("*")
            if img.is_file()
            and img.suffix.lower() in [
                ".jpg",
                ".jpeg",
                ".png"
                ]
            ]

        for img_path in images:

            image_paths.append(
                str(img_path)
            )

            labels.append(
                f"{crop.name}_{disease.name}"
            )

In [ ]:
print(
    "Total Images:",
    len(image_paths)
)

print(
    "Total Labels:",
    len(labels)
)

In [ ]:
print("Total Images:", len(image_paths))

In [ ]:
embeddings = []

valid_labels = []

valid_paths = []

failed_paths = []

for i, path in enumerate(
    tqdm(image_paths)
):

    try:

        emb = get_embedding(path)

        embeddings.append(emb)

        valid_labels.append(
            labels[i]
        )

        valid_paths.append(
            path
        )

    except Exception as e:

        failed_paths.append(
            (path, str(e))
        )

        print(
            f"\nSkipped: {path}"
        )

        continue

    if (i + 1) % 1000 == 0:

        np.save(
            "../embeddings/checkpoint_embeddings.npy",
            np.array(embeddings)
        )

        np.save(
            "../embeddings/checkpoint_labels.npy",
            np.array(valid_labels)
        )

        np.save(
            "../embeddings/checkpoint_paths.npy",
            np.array(valid_paths)
        )

        progress = ((i + 1) / len(image_paths)) * 100

print(
    f"Checkpoint saved: {i+1}/{len(image_paths)} ({progress:.2f}%)"
)

In [ ]:
embeddings = np.array(embeddings)

print(embeddings.shape)

In [ ]:
np.save(
    "../embeddings/embeddings.npy",
    embeddings
)

np.save(
    "../embeddings/labels.npy",
    np.array(valid_labels)
)

np.save(
    "../embeddings/paths.npy",
    np.array(valid_paths)
)

np.save(
    "../embeddings/failed_paths.npy",
    np.array(failed_paths, dtype=object)
)

In [8]:
import numpy as np

checkpoint_embeddings = np.load("../embeddings/checkpoint_embeddings.npy")
checkpoint_labels = np.load("../embeddings/checkpoint_labels.npy", allow_pickle=True)
checkpoint_paths = np.load("../embeddings/checkpoint_paths.npy", allow_pickle=True)

print("Checkpoint Embeddings:", checkpoint_embeddings.shape)
print("Checkpoint Labels:", checkpoint_labels.shape)
print("Checkpoint Paths:", checkpoint_paths.shape)

Checkpoint Embeddings: (2000, 2048)
Checkpoint Labels: (2000,)
Checkpoint Paths: (2000,)
